# 하이퍼파라미터 실험

설정을 하나씩 바꿔가며 돌리고 결과를 `outputs/tuning.csv`에 쌓는다.

**원칙**
- **한 번에 하나만 바꾼다.** 격자 탐색(모든 조합)은 5개 항목 × 3가지면 243번이라 불가능하다.
- **fold 0, 같은 seed로 고정**해서 비교한다. 다른 조건이 섞이면 비교가 무의미하다.
- 실험용은 정면만 + 1 fold + 적은 에폭으로 돌린다 (설정 1개당 약 1시간).

**먼저 할 일: 오차 범위 재기**

검증이 160명이라 우연히 0.02~0.03씩 흔들린다. 같은 설정을 seed만 바꿔 두 번 돌려서
그 폭을 먼저 확인해야, 나중에 "이 설정이 더 낫다"를 판단할 수 있다.

**판단 기준**
- 주 지표: `config.PRIMARY_TASK` (지금은 모공 개수)
- 보조: 전체 항목의 순위 상관 평균
- 주 지표가 오차 범위 안에서 유지되면서 평균이 오르면 채택

In [ ]:
import csv
import importlib
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import torch

import config
import data
import engine
import model as model_module

for module in (config, data, model_module, engine):
    importlib.reload(module)

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
config.OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = config.OUT_DIR / "tuning.csv"

print(f"장치 {config.DEVICE} | 백본 {config.BACKBONE}")
print(f"부위 {config.REGIONS}")
print(f"각도 {config.VIEWS} | 항목 {config.TASK_NAMES}")

In [ ]:
rows, test_rows = data.load_rows()
folds = engine.split_folds(rows)
train_rows, valid_rows = folds[0]  # 실험은 항상 fold 0으로 고정

print(f"학습 크롭 {len(rows):,}개 (사람 {len({r['subject'] for r in rows}):,}명)")
print(f"fold 0: 학습 {len(train_rows):,} / 검증 {len(valid_rows):,} "
      f"(사람 {len({r['subject'] for r in train_rows})} / {len({r['subject'] for r in valid_rows})})")
print(f"테스트 보관 {len(test_rows):,}개 (사람 {len({r['subject'] for r in test_rows}):,}명)")

counts = {task: sum(task in r["targets"] for r in rows) for task in config.TASK_NAMES}
print("\n항목별 학습 칸 수:", counts)

## 실험 돌리는 함수

`overrides`로 `config`의 값을 임시로 바꾼 뒤 학습하고, 끝나면 원래대로 되돌린다.

In [ ]:
def run_experiment(name, epochs=6, **overrides):
    """설정을 바꿔 fold 0을 학습하고 결과를 tuning.csv에 한 줄 추가한다."""
    original = {key: getattr(config, key) for key in overrides}
    for key, value in overrides.items():
        setattr(config, key, value)
    torch.manual_seed(config.SEED)

    print(f"\n=== {name} === {overrides}")
    started = time.time()
    try:
        best = engine.train_fold(train_rows, valid_rows, epochs=epochs)
    finally:
        for key, value in original.items():  # 다음 실험에 영향이 가지 않게 되돌린다
            setattr(config, key, value)

    result = dict(name=name, when=datetime.now().strftime("%m-%d %H:%M"), minutes=round((time.time() - started) / 60, 1),
                  epochs=epochs, best_epoch=best["epoch"], primary=round(best["score"], 4),
                  mean=round(engine.mean_spearman(best["results"]), 4),
                  **{task: round(r["spearman"], 4) for task, r in best["results"].items()},
                  **{k: str(v) for k, v in overrides.items()})

    is_new = not RESULT_PATH.exists()
    old = list(csv.DictReader(open(RESULT_PATH, newline=""))) if not is_new else []
    fields = list(dict.fromkeys([f for r in old for f in r] + list(result)))
    with open(RESULT_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(old + [result])

    print(f"→ {name}: 주지표 {result['primary']:.3f}, 평균 {result['mean']:.3f} ({result['minutes']:.0f}분)")
    return result


def show_results():
    if not RESULT_PATH.exists():
        print("아직 실험 결과가 없다.")
        return []
    results = list(csv.DictReader(open(RESULT_PATH, newline="")))
    print(f"{'실험':22s} {'주지표':>7s} {'평균':>7s} {'분':>5s}  변경한 설정")
    for r in results:
        changed = {k: v for k, v in r.items()
                   if k not in ("name", "when", "minutes", "epochs", "best_epoch", "primary", "mean")
                   and k not in config.TASK_NAMES and v}
        print(f"{r['name']:22s} {float(r['primary']):7.3f} {float(r['mean']):7.3f} {float(r['minutes']):5.0f}  {changed}")
    return results

## 0단계. 오차 범위 재기 (가장 먼저)

같은 설정을 seed만 바꿔 두 번 돌린다. 두 결과의 차이가 **잡음의 크기**다.
이후 실험에서 이 폭보다 작은 차이는 "차이 없음"으로 봐야 한다.

In [ ]:
run_experiment("기준", epochs=6)
run_experiment("기준(seed 43)", epochs=6, SEED=43)

results = show_results()
if len(results) >= 2:
    gap = abs(float(results[-1]["primary"]) - float(results[-2]["primary"]))
    print(f"\n같은 설정인데 seed만 달라서 생긴 차이: {gap:.3f}")
    print(f"→ 앞으로 {gap:.3f} 이내의 차이는 의미 없는 것으로 본다.")

## 1단계. 학습률 (영향이 가장 큼)

백본은 이미 잘 배운 상태라 작게, 헤드는 랜덤에서 시작하니 크게 준다. 그 비율과 크기를 본다.

In [ ]:
run_experiment("학습률 낮게", epochs=6, LR_BACKBONE=1e-5, LR_HEAD=1e-4)
run_experiment("학습률 높게", epochs=6, LR_BACKBONE=5e-5, LR_HEAD=5e-4)
show_results();

## 2단계. 백본을 얼마나 얼릴지

데이터가 800명뿐이라 많이 얼리는 쪽이 유리할 수 있다.
`0`=거의 전부 학습, `2`=앞 4.4% 고정(지금), `3`=앞 44% 고정.

In [ ]:
run_experiment("동결 없음", epochs=6, FREEZE_STAGES=0)
run_experiment("동결 3단계", epochs=6, FREEZE_STAGES=3)
show_results();

## 3단계. 증강 강도

약하면 외우고, 강하면 신호가 망가진다. 흐림·강한 색 변화는 모공·색소를 해쳐서 후보에서 뺐다.

In [ ]:
run_experiment("증강 약하게", epochs=6, AUG_ROTATE=5, AUG_SCALE=0.02, AUG_BRIGHT=0.05)
run_experiment("증강 세게", epochs=6, AUG_ROTATE=20, AUG_SCALE=0.10, AUG_BRIGHT=0.20)
show_results();

## 4단계. 헤드 구조 (영향이 가장 작음)

헤드는 전체 파라미터의 0.7%라, 대개 차이가 오차 범위 안에 들어간다.
앞 단계가 끝나고 시간이 남으면 확인하는 정도로 충분하다.

In [ ]:
run_experiment("헤드 1층", epochs=6, HEAD_HIDDEN=0)
run_experiment("헤드 512", epochs=6, HEAD_HIDDEN=512)
run_experiment("드롭아웃 0.4", epochs=6, DROPOUT=0.4)
show_results();

## 결과 보기

In [ ]:
results = show_results()
if results:
    names = [r["name"] for r in results]
    primary = [float(r["primary"]) for r in results]
    mean = [float(r["mean"]) for r in results]
    baseline = primary[0]

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
    x = np.arange(len(names))
    axes[0].bar(x - 0.2, primary, 0.4, label=f"주지표({config.PRIMARY_TASK})")
    axes[0].bar(x + 0.2, mean, 0.4, label="전체 평균")
    axes[0].axhline(baseline, color="red", ls="--", lw=1, label="기준 실험")
    axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=30, ha="right")
    axes[0].set_ylabel("Spearman"); axes[0].legend(); axes[0].grid(alpha=.3, axis="y")

    tasks = [t for t in config.TASK_NAMES if t in results[0]]
    for r in results:
        axes[1].plot(tasks, [float(r[t]) for t in tasks], marker="o", label=r["name"])
    axes[1].set_title("항목별 성능"); axes[1].set_ylabel("Spearman")
    axes[1].tick_params(axis="x", rotation=30); axes[1].grid(alpha=.3); axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()